# Celestial Gardeners' Guild — Manual Trading 3

IMC Prosperity 4 manual trading challenge solver.

---

## What is this notebook?

This is a **math-powered cheat sheet** for a trading game. In the game, you're buying items from a bunch of sellers (called "counterparties"). Each seller has a secret minimum price they'll accept (their **reserve price**). You don't know any individual seller's reserve price, but you *do* know the overall pattern of how those prices are spread out.

Your goal: **pick two bid prices that maximize your total profit.**

We'll build up from simple to complex:
- **Part A**: What if you could only place one bid?
- **Part B**: What's the best pair of bids if you ignore competition?
- **Part C**: How do other players' bids affect yours? (Game theory)
- **Part D**: Stress-test our answer with random simulations
- **Part E**: How sensitive is our answer to our assumptions?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import collections
from itertools import product as iterproduct

## Problem Statement

### The Setup (plain English)

Imagine a flea market with lots of sellers. Each seller has one item of **SCUBA_GEAR** to sell. You can resell any gear you buy tomorrow for exactly **920 SeaShells** (the "fair price").

You submit **two bids** — a low bid ($b_1$) and a high bid ($b_2$). Both must be multiples of 5, between 670 and 920. And $b_1$ must be less than $b_2$.

### Key vocabulary

| Term | Meaning |
|------|---------|
| **Reserve price** ($r$) | The secret minimum a seller will accept. If your bid is below this, they say no. |
| **Bid** ($b_1$, $b_2$) | The prices you offer. $b_1$ is your "lowball" and $b_2$ is your "reach" bid. |
| **Fair price** (920) | What you can resell the gear for tomorrow. Your profit on any trade = 920 minus what you paid. |
| **Grid** | Bids and reserve prices live on {670, 675, 680, ..., 915, 920} — multiples of 5. |
| **Field / $\bar{b}_2$** | The average second bid of all *other* players in the game. You don't know this — you have to guess. |

### How trades happen (the 4 rules)

For each seller with reserve price $r$:

1. **$b_1 \geq r$** → You buy at your low bid. Profit = $920 - b_1$. (Great deal — you got it cheap!)

2. **$b_1 < r \leq b_2$ AND your $b_2$ beats the field** ($b_2 > \bar{b}_2$) → You buy at $b_2$. Profit = $920 - b_2$. (Clean trade — you outbid everyone else.)

3. **$b_1 < r \leq b_2$ AND your $b_2$ does NOT beat the field** ($b_2 \leq \bar{b}_2$) → You still buy at $b_2$, but your profit gets **penalized** by a harsh multiplier: $\left(\frac{920 - \bar{b}_2}{920 - b_2}\right)^3$. This shrinks your profit because you're not competitive.

4. **$r > b_2$** → No trade. The seller wanted more than either of your bids.

### Why the penalty matters

The penalty in rule 3 is **cubic** (raised to the power of 3). That means if you're even a little below the field average, your profit gets crushed. For example, if the field averages 890 and you bid 880:
- Multiplier = $((920-890)/(920-880))^3 = (30/40)^3 = 0.42$
- You only keep 42% of what you'd normally earn!

This is the game's way of saying: "if you're not competitive on your high bid, you get punished."

### The distribution of reserve prices

Reserve prices are **uniformly distributed** on the grid {670, 675, ..., 920}.

**What does "uniformly distributed" mean?** Every value on the grid is equally likely. There are 51 possible values, so each has a 1/51 chance (~1.96%). A seller is just as likely to want 670 as 920 or anything in between.

## Grid & Distribution

Below we define the **grid** — all the valid prices — and plot the distribution.

Since every reserve price is equally likely, the bar chart will be flat (every bar the same height). This is what "uniform" looks like visually.

In [ ]:
# Bid grid
GRID = np.arange(670, 921, 5)  # 670, 675, ..., 920
FAIR = 920
N_GRID = len(GRID)
print(f"Grid: {GRID[0]} to {GRID[-1]}, {N_GRID} values, step 5")

# Reserve prices are uniform on GRID => each has probability 1/51
P_EACH = 1.0 / N_GRID

fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(GRID, [P_EACH] * N_GRID, width=4, color='steelblue', alpha=0.7)
ax.set_xlabel('Reserve price')
ax.set_ylabel('Probability')
ax.set_title('Uniform distribution of reserve prices')
plt.tight_layout()
plt.show()

## Part A: Solve $b_1$ in Isolation

### The idea

Forget about $b_2$ for now. Pretend you can only place **one** bid. What should it be?

There's a fundamental **tradeoff**:
- **Bid high** (e.g., 910): Almost every seller accepts, so you buy lots of items. But your profit per item is tiny (920 - 910 = 10 each).
- **Bid low** (e.g., 700): Huge profit per item (920 - 700 = 220 each). But very few sellers will accept because most have reserve prices above 700.

The sweet spot is somewhere in the middle.

### The math (step by step)

**Expected profit** = (chance a seller accepts) $\times$ (profit if they accept)

- **Chance a seller accepts**: If you bid $b_1$, a seller accepts when $r \leq b_1$. Since reserve prices are uniform on the grid, this probability is just "how many grid values are $\leq b_1$" divided by 51.
  - Example: if $b_1 = 795$, there are 26 grid values from 670 to 795, so the chance is 26/51 $\approx$ 51%.

- **Profit if they accept**: $920 - b_1$
  - Example: if $b_1 = 795$, profit per trade = 125.

- **Expected profit per seller** = $(26/51) \times 125 \approx 63.7$

### What is "expected value"?

**Expected value** is the average outcome if you repeated the experiment many times. If you faced 51,000 sellers and bid 795, you'd expect roughly 26,000 trades at 125 profit each. The expected profit *per seller* is 63.7.

We try every possible bid on the grid and pick the one with the highest expected profit.

In [ ]:
def prob_leq(b):
    """Probability that reserve price r <= b, on the discrete grid."""
    count = np.sum(GRID <= b)
    return count / N_GRID

def prob_between(lo, hi):
    """Probability that lo < r <= hi, on the discrete grid."""
    count = np.sum((GRID > lo) & (GRID <= hi))
    return count / N_GRID

# Grid search for b1 alone
profits_b1 = np.array([(FAIR - b) * prob_leq(b) for b in GRID])
best_idx = np.argmax(profits_b1)
best_b1_alone = GRID[best_idx]

print(f"Optimal b1 (isolation): {best_b1_alone}")
print(f"Expected profit per counterparty: {profits_b1[best_idx]:.2f}")
print(f"Prob(trade): {prob_leq(best_b1_alone):.3f}, margin: {FAIR - best_b1_alone}")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(GRID, profits_b1, 'o-', markersize=4, color='steelblue')
ax.axvline(best_b1_alone, color='red', linestyle='--', label=f'Optimal b1 = {best_b1_alone}')
ax.set_xlabel('b1')
ax.set_ylabel('Expected profit per counterparty')
ax.set_title('Part A: Single-bid expected profit')
ax.legend()
plt.tight_layout()
plt.show()

## Part B: Joint $(b_1, b_2)$ Optimization — No Penalty

### The idea

Now we use **both** bids. Think of it as casting two nets:
- $b_1$ (low bid) catches all the "cheap" sellers (reserve price $\leq b_1$). You pay $b_1$ for each.
- $b_2$ (high bid) catches sellers that $b_1$ missed but who are still willing at $b_2$ (reserve price between $b_1$ and $b_2$). You pay $b_2$ for each.

For now, we **ignore the penalty** — we pretend our $b_2$ always beats the field (rule 2, never rule 3).

### The math

Expected profit = (profit from $b_1$ trades) + (profit from $b_2$ trades)

$$\underbrace{\Pr(r \leq b_1) \times (920 - b_1)}_{\text{low-bid catches}} + \underbrace{\Pr(b_1 < r \leq b_2) \times (920 - b_2)}_{\text{high-bid catches the rest}}$$

### What is a "heatmap"?

We try **every valid combination** of $(b_1, b_2)$ — that's about 1,275 pairs. For each pair, we compute the expected profit. A **heatmap** colors each pair by its profit: bright = good, dark = bad. It lets you see at a glance where the best region is.

### What is a "grid search"?

Instead of using calculus to find the best answer analytically, we just **try every possibility** and keep the winner. With only 1,275 pairs, this is instant on a computer. This brute-force approach is called a **grid search**.

In [ ]:
def expected_profit_no_penalty(b1, b2):
    """Expected profit per counterparty, ignoring the b2 penalty."""
    return (FAIR - b1) * prob_leq(b1) + (FAIR - b2) * prob_between(b1, b2)

# Vectorized grid search
profit_matrix = np.full((N_GRID, N_GRID), np.nan)
best_val = -np.inf
best_pair = (GRID[0], GRID[0])

for i, b1 in enumerate(GRID):
    for j, b2 in enumerate(GRID):
        if b2 <= b1:
            continue
        val = expected_profit_no_penalty(b1, b2)
        profit_matrix[i, j] = val
        if val > best_val:
            best_val = val
            best_pair = (b1, b2)

print(f"Optimal (b1, b2) [no penalty]: {best_pair}")
print(f"Expected profit per counterparty: {best_val:.2f}")

# Heatmap
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(profit_matrix, origin='lower', aspect='auto',
               extent=[GRID[0], GRID[-1], GRID[0], GRID[-1]],
               cmap='viridis')
ax.set_xlabel('b2 (high bid)')
ax.set_ylabel('b1 (low bid)')
ax.set_title('Part B: Expected profit heatmap (no penalty)')
ax.plot(best_pair[1], best_pair[0], 'r*', markersize=15, label=f'Optimum {best_pair}')
ax.legend()
plt.colorbar(im, label='Expected profit per counterparty')
plt.tight_layout()
plt.show()

## Part C: Level-k Reasoning for $b_2$

### The problem: you don't know what others will bid

The penalty in rule 3 depends on $\bar{b}_2$ — the **average** of everyone else's second bid. You don't know this! So we need to *guess* what others will do.

### What is "level-k reasoning"?

This is a technique from **game theory** (the math of strategic decision-making). It works like this:

- **Level-0 player**: Totally naive. Ignores the penalty. Just optimizes as if rule 3 doesn't exist (= our Part B answer).

- **Level-1 player**: Thinks "everyone else is level-0." So they assume $\bar{b}_2$ = the level-0 answer. Then they pick the best $(b_1, b_2)$ given that assumption.

- **Level-2 player**: Thinks "everyone else is level-1." So they assume $\bar{b}_2$ = the level-1 answer. Then they best-respond to *that*.

- **Level-k player**: Assumes everyone else is level-(k-1), and best-responds.

### Why does this work?

It's like a **chain of reasoning**:
> "If I think you're naive, I should do X. But if you think I'm naive and react, I should do Y instead. But if you anticipate that..."

This chain usually **converges** — after a few steps, the answer stops changing. That's the **equilibrium** (a stable point where nobody wants to deviate).

### What is "best-respond"?

**Best response** = the strategy that maximizes your profit, *given a fixed assumption about what others do*. It's not about being "best" in general — it's about being best *conditional on your belief about the opponent*.

### What is "convergence"?

When the level-k answers stop changing (level-5 gives the same answer as level-4), we say the process has **converged**. The answer has stabilized.

### What is a "cycle"?

Sometimes instead of converging, the answers **oscillate** — level-3 gives the same answer as level-1, then level-4 matches level-2, etc. We detect this and stop.

In [ ]:
def expected_profit_with_penalty(b1, b2, avg_b2):
    """Expected profit per counterparty, accounting for the b2 penalty.
    
    Parameters
    ----------
    b1 : int
        Low bid.
    b2 : int
        High bid.
    avg_b2 : float
        Assumed average second bid of the field.
    
    Returns
    -------
    float
        Expected profit per counterparty.
    """
    profit_b1 = (FAIR - b1) * prob_leq(b1)
    p_b2_range = prob_between(b1, b2)
    
    if b2 > avg_b2:
        # Clean trade — no penalty
        profit_b2 = (FAIR - b2) * p_b2_range
    else:
        # Penalized: multiplier = ((920 - avg_b2) / (920 - b2))^3
        if FAIR - b2 == 0:
            profit_b2 = 0.0
        else:
            penalty_multiplier = ((FAIR - avg_b2) / (FAIR - b2)) ** 3
            profit_b2 = (FAIR - b2) * penalty_multiplier * p_b2_range
    
    return profit_b1 + profit_b2


def best_respond(avg_b2):
    """Find the (b1, b2) pair that maximizes expected profit given assumed avg_b2.
    
    Parameters
    ----------
    avg_b2 : float
        Assumed average second bid of the field.
    
    Returns
    -------
    tuple
        (best_b1, best_b2, best_profit)
    """
    best_val = -np.inf
    best = (GRID[0], GRID[-1])
    for b1 in GRID:
        for b2 in GRID:
            if b2 <= b1:
                continue
            val = expected_profit_with_penalty(b1, b2, avg_b2)
            if val > best_val:
                best_val = val
                best = (b1, b2)
    return best[0], best[1], best_val


# Level-k tower
MAX_LEVELS = 10
levels = []

# Level 0: no penalty consideration
level0_b1, level0_b2 = best_pair
level0_profit = best_val
levels.append({'level': 0, 'b1': level0_b1, 'b2': level0_b2, 
               'profit': round(level0_profit, 4), 'avg_b2_assumed': 'N/A'})

prev_b2 = float(level0_b2)
for k in range(1, MAX_LEVELS + 1):
    b1_k, b2_k, profit_k = best_respond(prev_b2)
    levels.append({'level': k, 'b1': b1_k, 'b2': b2_k,
                   'profit': round(profit_k, 4), 'avg_b2_assumed': prev_b2})
    
    # Check convergence
    if b2_k == prev_b2:
        print(f"Converged at level {k}")
        break
    
    # Check cycle
    past_b2s = [l['b2'] for l in levels[:-1]]
    if b2_k in past_b2s:
        print(f"Cycle detected at level {k}: b2={b2_k} appeared before")
        break
    
    prev_b2 = float(b2_k)

# Print table
print(f"\n{'Level':<7} {'b1':<6} {'b2':<6} {'Assumed avg_b2':<16} {'E[profit]':<10}")
print('-' * 50)
for l in levels:
    avg_str = f"{l['avg_b2_assumed']:.0f}" if isinstance(l['avg_b2_assumed'], float) else l['avg_b2_assumed']
    print(f"{l['level']:<7} {l['b1']:<6} {l['b2']:<6} {avg_str:<16} {l['profit']:<10.4f}")

## Part D: Monte Carlo Robustness Check

### What is Monte Carlo simulation?

Named after the famous casino, **Monte Carlo** means: instead of computing the exact answer with math, we **simulate the random process thousands of times** and look at the distribution of outcomes.

Think of it like this: Parts A-C computed the *average* profit. But in the real game, you face one specific set of 5,000 sellers with random reserve prices. You might get lucky (lots of low reserve prices) or unlucky. Monte Carlo tells us **how much the actual profit might vary**.

### What we do here

1. **Generate 5,000 random reserve prices** from the uniform distribution (simulating one game).
2. **Compute total profit** for a given $(b_1, b_2)$.
3. **Repeat 1,000 times** to get 1,000 different possible profits.
4. Plot the results as a **histogram** (a bar chart showing how often each profit level occurred).

### What is a "field mixture"?

We still don't know the field's average $b_2$. So we **model the other players** as a mix:
- 30% are level-0 (naive, ignore penalty)
- 40% are level-1 (one step of reasoning)
- 20% are level-2 (two steps)
- 10% are "naive" (just bid 920, the maximum)

The weights are defined at the top of the code cell — **tweak them** to see how results change.

### What do the stats mean?

| Stat | Meaning |
|------|---------|
| **Mean PnL** | Average total profit across 1,000 simulations. The "expected" outcome. |
| **Std** | **Standard deviation** — how spread out the results are. Low = consistent. High = risky. |
| **5th %ile** | The profit you'd beat 95% of the time. Your "bad luck" scenario. |
| **95th %ile** | The profit you'd only beat 5% of the time. Your "good luck" scenario. |

In [ ]:
# ====== MIXTURE WEIGHTS — EASY TO TWEAK ======
# Maps level -> weight. Weights are normalized automatically.
FIELD_MIXTURE = {
    0: 0.30,   # 30% play level-0 (naive, ignore penalty)
    1: 0.40,   # 40% play level-1
    2: 0.20,   # 20% play level-2
    'naive': 0.10,  # 10% bid naively at 920 (maximum)
}
# =============================================

def field_avg_b2(mixture, levels_table):
    """Compute the field's average b2 given a mixture of level-k players.
    
    Parameters
    ----------
    mixture : dict
        Maps level (int or 'naive') to weight.
    levels_table : list of dict
        Level-k tower results.
    
    Returns
    -------
    float
        Weighted average b2 of the field.
    """
    level_b2 = {l['level']: l['b2'] for l in levels_table}
    total_w = sum(mixture.values())
    avg = 0.0
    for k, w in mixture.items():
        if k == 'naive':
            avg += (w / total_w) * 920
        elif k in level_b2:
            avg += (w / total_w) * level_b2[k]
        else:
            # Use highest available level
            max_level = max(l['level'] for l in levels_table)
            avg += (w / total_w) * level_b2[max_level]
    return avg


def simulate_profit(b1, b2, avg_b2, n_counterparties, n_sims):
    """Monte Carlo simulation of total profit.
    
    Parameters
    ----------
    b1 : int
        Low bid.
    b2 : int
        High bid.
    avg_b2 : float
        Assumed field average b2.
    n_counterparties : int
        Number of counterparties per simulation.
    n_sims : int
        Number of simulations.
    
    Returns
    -------
    ndarray
        Array of total profits, shape (n_sims,).
    """
    # Sample reserve prices: uniform on GRID
    reserves = np.random.choice(GRID, size=(n_sims, n_counterparties))
    
    # Trades at b1
    mask_b1 = reserves <= b1
    profit_b1 = mask_b1.astype(float) * (FAIR - b1)
    
    # Trades at b2
    mask_b2 = (reserves > b1) & (reserves <= b2)
    if b2 > avg_b2:
        per_trade_b2 = FAIR - b2
    else:
        if FAIR - b2 == 0:
            per_trade_b2 = 0.0
        else:
            per_trade_b2 = (FAIR - b2) * ((FAIR - avg_b2) / (FAIR - b2)) ** 3
    profit_b2 = mask_b2.astype(float) * per_trade_b2
    
    return (profit_b1 + profit_b2).sum(axis=1)


# Compute field avg_b2
avg_b2_field = field_avg_b2(FIELD_MIXTURE, levels)
print(f"Field average b2 (mixture model): {avg_b2_field:.1f}")

# Candidates from the level-k tower
candidates = list({(l['b1'], l['b2']) for l in levels})
candidates.sort()

N_COUNTERPARTIES = 5000
N_SIMS = 1000
np.random.seed(42)

fig, axes = plt.subplots(len(candidates), 1, figsize=(12, 3 * len(candidates)), sharex=True)
if len(candidates) == 1:
    axes = [axes]

print(f"\n{'(b1, b2)':<16} {'Mean PnL':>10} {'Std':>10} {'5th %ile':>10} {'95th %ile':>10}")
print('-' * 60)

for ax, (b1, b2) in zip(axes, candidates):
    pnls = simulate_profit(b1, b2, avg_b2_field, N_COUNTERPARTIES, N_SIMS)
    mean_pnl = pnls.mean()
    std_pnl = pnls.std()
    p5, p95 = np.percentile(pnls, [5, 95])
    
    print(f"({b1}, {b2}){'':<8} {mean_pnl:>10.0f} {std_pnl:>10.0f} {p5:>10.0f} {p95:>10.0f}")
    
    ax.hist(pnls, bins=40, alpha=0.7, color='steelblue', edgecolor='white')
    ax.axvline(mean_pnl, color='red', linestyle='--', label=f'Mean = {mean_pnl:.0f}')
    ax.set_title(f'(b1={b1}, b2={b2})')
    ax.legend()

axes[-1].set_xlabel('Total PnL')
fig.suptitle(f'Monte Carlo PnL distributions (n={N_COUNTERPARTIES}, {N_SIMS} sims)', y=1.01)
plt.tight_layout()
plt.show()

## Part E: Sensitivity Analysis

### What is sensitivity analysis?

Our whole strategy depends on an assumption we made up: the "field mixture" (what percentage of players are naive vs. sophisticated). **Sensitivity analysis** asks: "if that assumption is wrong, how badly does it mess up our answer?"

If the optimal bids barely change across wildly different assumptions, our answer is **robust** (trustworthy). If the bids swing wildly, we should be cautious.

### The scenarios

We test several extreme cases:
- **"Mostly naive"**: Most players don't think about the penalty at all.
- **"Balanced"**: A mix of sophistication levels (our base case).
- **"Sophisticated"**: Most players are doing level-2+ reasoning.
- **"All naive"**: Everyone just bids 920 (no strategic thinking).
- **"All level-0"**: Everyone does Part B optimization but ignores the penalty.

For each scenario, we re-solve the optimization and see what $(b_1, b_2)$ comes out.

In [ ]:
def optimal_for_mixture(mixture, levels_table):
    """Find optimal (b1, b2) given a field mixture.
    
    Parameters
    ----------
    mixture : dict
        Maps level (int or 'naive') to weight.
    levels_table : list of dict
        Level-k tower results.
    
    Returns
    -------
    tuple
        (best_b1, best_b2, best_profit, avg_b2)
    """
    avg = field_avg_b2(mixture, levels_table)
    best_val = -np.inf
    best = (GRID[0], GRID[-1])
    for b1 in GRID:
        for b2 in GRID:
            if b2 <= b1:
                continue
            val = expected_profit_with_penalty(b1, b2, avg)
            if val > best_val:
                best_val = val
                best = (b1, b2)
    return best[0], best[1], best_val, avg


# Define mixture scenarios
scenarios = [
    ('Mostly naive',      {0: 0.5, 1: 0.2, 2: 0.1, 'naive': 0.2}),
    ('Balanced',          {0: 0.3, 1: 0.4, 2: 0.2, 'naive': 0.1}),
    ('Sophisticated',     {0: 0.1, 1: 0.3, 2: 0.4, 'naive': 0.2}),
    ('Very sophisticated',{0: 0.05, 1: 0.15, 2: 0.5, 'naive': 0.3}),
    ('All naive',         {0: 0.0, 1: 0.0, 2: 0.0, 'naive': 1.0}),
    ('All level-0',       {0: 1.0, 1: 0.0, 2: 0.0, 'naive': 0.0}),
]

print(f"{'Scenario':<22} {'b1':<6} {'b2':<6} {'Field avg_b2':<14} {'E[profit]':<10}")
print('-' * 60)
results_sensitivity = []
for name, mix in scenarios:
    b1, b2, profit, avg = optimal_for_mixture(mix, levels)
    print(f"{name:<22} {b1:<6} {b2:<6} {avg:<14.1f} {profit:<10.4f}")
    results_sensitivity.append((name, b1, b2, profit, avg))

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

names = [r[0] for r in results_sensitivity]
b1s = [r[1] for r in results_sensitivity]
b2s = [r[2] for r in results_sensitivity]
profits = [r[3] for r in results_sensitivity]

x = np.arange(len(names))
ax1.bar(x - 0.15, b1s, 0.3, label='b1', color='steelblue')
ax1.bar(x + 0.15, b2s, 0.3, label='b2', color='coral')
ax1.set_xticks(x)
ax1.set_xticklabels(names, rotation=30, ha='right')
ax1.set_ylabel('Bid value')
ax1.set_title('Optimal bids by scenario')
ax1.legend()

ax2.bar(x, profits, color='seagreen')
ax2.set_xticks(x)
ax2.set_xticklabels(names, rotation=30, ha='right')
ax2.set_ylabel('Expected profit per counterparty')
ax2.set_title('Expected profit by scenario')

plt.tight_layout()
plt.show()

## Final Recommendation

### Summary — what we learned in each part

| Part | Question | Key insight |
|------|----------|-------------|
| **A** | Best single bid? | ~795 — the midpoint between "buy everything cheaply" and "buy nothing expensively" |
| **B** | Best pair ignoring competition? | Adding $b_2$ captures sellers that $b_1$ missed, boosting total profit |
| **C** | How does competition change things? | The cubic penalty pushes $b_2$ higher — you want to beat the field average |
| **D** | How variable is the outcome? | With 5,000 sellers, outcomes are tightly clustered around the expected value (law of large numbers) |
| **E** | Does our answer depend on assumptions? | $b_1$ is robust; $b_2$ shifts a bit but nearby values have similar profit |

### Strategy

- **$b_1$**: Use the level-k result from Part C. This is robust across assumptions.
- **$b_2$**: Use the level-1 or level-2 result. When uncertain, **bid slightly higher** — the cubic penalty for being below the mean is much worse than the linear cost of slightly overpaying.

### Why "slightly higher" is safer

- **Being 10 too high** on $b_2$: You lose 10 profit per trade (linear, mild).
- **Being 10 too low** on $b_2$: The cubic penalty kicks in and can slash your profit by 50%+ (catastrophic).

The downside of being low is much worse than the downside of being high. When in doubt, round up.